# Notebook 03 — Leakage-safe split


In [1]:
# ── Cell 1 · Configuration ────────────────────────────────────────────────────
CONFIG = {
    "SEED": 42,
    "SPLIT": {"train": 0.70, "val": 0.15, "test": 0.15},


    "ALT_SPLITS": [(0.80, 0.10, 0.10), (0.60, 0.20, 0.20)],

    "USE_DRIVE":      True,
    "DRIVE_DIR":      "/content/drive/MyDrive/ClinicalShield_v2",
    "PUSH_TO_GITHUB": True,
    "GITHUB_REPO":    "NehlTech/ClinicalShield",
    "GITHUB_BRANCH":  "v2-revision",
    "GIT_USER_NAME":  "Adu-Boahene Bright",
    "GIT_USER_EMAIL": "baduboahene@st.knust.edu.gh",
}
SEED = CONFIG["SEED"]
print("split:", CONFIG["SPLIT"], "| seed:", SEED)


split: {'train': 0.7, 'val': 0.15, 'test': 0.15} | seed: 42


In [2]:
# ── Cell 2 · Environment and inputs ──────────────────────────────────────
import sys, os, json, random, hashlib, subprocess, shutil, time
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

random.seed(SEED); np.random.seed(SEED)

IN_COLAB = "google.colab" in sys.modules
DRIVE_ROOT, REPO_DIR = None, None

if IN_COLAB:
    if CONFIG["USE_DRIVE"]:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = Path(CONFIG["DRIVE_DIR"]); DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        print("drive :", DRIVE_ROOT)
    repo_name = CONFIG["GITHUB_REPO"].split("/")[-1]
    REPO_DIR = Path("/content") / repo_name
    if not REPO_DIR.exists():
        try:
            from google.colab import userdata
            tok = userdata.get("GITHUB_TOKEN")
            r = subprocess.run(["git","clone","-q",
                "https://" + tok + "@github.com/" + CONFIG["GITHUB_REPO"] + ".git",
                str(REPO_DIR)], capture_output=True, text=True)
            print("clone :", "ok" if r.returncode == 0 else r.stderr[:200])
        except Exception as e:
            print("clone skipped:", type(e).__name__)
    else:
        print("clone : already present")
    if REPO_DIR.exists():
        for k, v in [("user.name", CONFIG["GIT_USER_NAME"]),
                     ("user.email", CONFIG["GIT_USER_EMAIL"])]:
            subprocess.run(["git","-C",str(REPO_DIR),"config",k,v], check=False)
        subprocess.run(["git","-C",str(REPO_DIR),"checkout","-q",
                        CONFIG["GITHUB_BRANCH"]], check=False, capture_output=True)
        subprocess.run(["git","-C",str(REPO_DIR),"pull","-q","origin",
                        CONFIG["GITHUB_BRANCH"]], check=False, capture_output=True)
    ROOT = REPO_DIR if REPO_DIR.exists() else Path("/content")
else:
    ROOT = Path.cwd()
    while not (ROOT/".git").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT/".git").exists():
        ROOT = Path.cwd()

DIRS = {"dataset": ROOT/"data"/"dataset", "stats": ROOT/"data"/"stats",
        "splits": ROOT/"data"/"splits"}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)
print("root  :", ROOT)

def read_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

def restore(rel_path, label):
    local = ROOT / rel_path
    if local.exists():
        return local, "repo"
    if DRIVE_ROOT:
        src = DRIVE_ROOT / rel_path
        if src.exists():
            local.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, local)
            return local, "drive"
    drive_msg = str(DRIVE_ROOT / rel_path) if DRIVE_ROOT else "(drive not mounted)"
    raise FileNotFoundError("\n" + label + " not found. Looked in:\n  repo : "
        + str(local) + "\n  drive: " + drive_msg + "\nRun NB02 first.")

p_ds, src_ds = restore("data/dataset/attack_dataset.jsonl", "attack_dataset.jsonl")
p_st, src_st = restore("data/stats/attack_stats.json",      "attack_stats.json")

dataset   = read_jsonl(p_ds)
nb02_stats = json.load(open(p_st))

exp_n = nb02_stats["counts"]["chunks"]
assert len(dataset) == exp_n, "row count %d != NB02 stats %d" % (len(dataset), exp_n)
exp_adv = nb02_stats["counts"]["adversarial"]
act_adv = sum(d["label"] for d in dataset)
assert act_adv == exp_adv, "adversarial %d != NB02 stats %d" % (act_adv, exp_adv)

print("")
print("attack_dataset.jsonl from " + src_ds + "  " + str(len(dataset)) + " chunks")
print("groups      : %d" % len({d["group_id"] for d in dataset}))
print("adversarial : %d (%.1f%%)" % (act_adv, 100*act_adv/len(dataset)))
print("NB02 inputs verified")


Mounted at /content/drive
drive : /content/drive/MyDrive/ClinicalShield_v2
clone : ok
root  : /content/ClinicalShield

attack_dataset.jsonl from drive  5353 chunks
groups      : 1487
adversarial : 2408 (45.0%)
NB02 inputs verified


In [3]:
# ── Cell 3 ───────────────────────────────────────────────


try:
    from sklearn.model_selection import StratifiedGroupKFold
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"])
    from sklearn.model_selection import StratifiedGroupKFold

y      = np.array([d["label"] for d in dataset])
groups = np.array([d["group_id"] for d in dataset])
X      = np.arange(len(dataset)).reshape(-1, 1)


sgkf1 = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)
folds = list(sgkf1.split(X, y, groups))

test_frac = CONFIG["SPLIT"]["test"]
val_frac  = CONFIG["SPLIT"]["val"]
n_hold    = int(round((test_frac + val_frac) * 10))

hold_idx = np.concatenate([folds[i][1] for i in range(n_hold)])
train_idx = np.setdiff1d(np.arange(len(dataset)), hold_idx)


hold_groups = groups[hold_idx]
hold_y      = y[hold_idx]
sgkf2 = StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=SEED)
a_rel, b_rel = next(iter(sgkf2.split(hold_idx.reshape(-1,1), hold_y, hold_groups)))
val_idx  = hold_idx[a_rel]
test_idx = hold_idx[b_rel]

split_of = {}
for i in train_idx: split_of[i] = "train"
for i in val_idx:   split_of[i] = "val"
for i in test_idx:  split_of[i] = "test"

for i, d in enumerate(dataset):
    d["split"] = split_of[i]

parts = {s: [d for d in dataset if d["split"] == s] for s in ["train","val","test"]}
print("partition    chunks   groups   adversarial")
for s in ["train","val","test"]:
    p = parts[s]
    g = len({d["group_id"] for d in p})
    a = sum(d["label"] for d in p)
    print("  %-8s %6d   %6d   %5d (%.1f%%)" % (s, len(p), g, a, 100*a/len(p)))


partition    chunks   groups   adversarial
  train      3787     1024    1710 (45.2%)
  val         759      228     337 (44.4%)
  test        807      235     361 (44.7%)


In [4]:
# ── Cell 4─────────────────────────────────


g2s = defaultdict(set)
for d in dataset:
    g2s[d["group_id"]].add(d["split"])

spanning = {g: s for g, s in g2s.items() if len(s) > 1}
print("CHECK 1 — no group spans partitions")
print("  groups total         : %d" % len(g2s))
print("  groups spanning >1   : %d" % len(spanning))
if spanning:
    for g, s in list(spanning.items())[:5]:
        print("    %s -> %s" % (g, sorted(s)))
print("  -> %s" % ("PASS" if not spanning else "FAIL"))
assert not spanning, "group leakage detected across partitions"


d2s = defaultdict(set)
for d in dataset:
    d2s[d["parent_doc_id"]].add(d["split"])
doc_span = {k: v for k, v in d2s.items() if len(v) > 1}
print("")
print("CHECK 2 — no source document spans partitions")
print("  documents spanning >1: %d  -> %s" % (len(doc_span), "PASS" if not doc_span else "FAIL"))
assert not doc_span, "source document leakage detected"


assert len(split_of) == len(dataset), "unassigned chunks"
counts = Counter(d["split"] for d in dataset)
assert sum(counts.values()) == len(dataset)
print("")
print("CHECK 3 — assignment completeness")
print("  chunks assigned      : %d/%d -> PASS" % (sum(counts.values()), len(dataset)))


CHECK 1 — no group spans partitions
  groups total         : 1487
  groups spanning >1   : 0
  -> PASS

CHECK 2 — no source document spans partitions
  documents spanning >1: 0  -> PASS

CHECK 3 — assignment completeness
  chunks assigned      : 5353/5353 -> PASS


In [5]:
# ── Cell 5 ───────────────────────────────────────────


print("label balance by partition")
for s in ["train","val","test"]:
    p = parts[s]
    a = sum(d["label"] for d in p)
    print("  %-6s adversarial %.1f%%" % (s, 100*a/len(p)))

overall = 100*sum(y)/len(y)
drift = max(abs(100*sum(d["label"] for d in parts[s])/len(parts[s]) - overall)
            for s in ["train","val","test"])
print("  max drift from overall %.1f%% : %.2f pp -> %s"
      % (overall, drift, "PASS" if drift < 3 else "REVIEW"))

for field in ["vector", "category", "encoding"]:
    print("")
    print("%s coverage in test" % field)
    all_vals = sorted({str(d[field]) for d in dataset if d["label"] == 1})
    tst = Counter(str(d[field]) for d in parts["test"] if d["label"] == 1)
    for v in all_vals:
        n = tst.get(v, 0)
        flag = "" if n >= 20 else "   <-- thin"
        print("  %-28s %5d%s" % (v, n, flag))
    missing = [v for v in all_vals if tst.get(v, 0) == 0]
    if missing:
        print("  MISSING FROM TEST: %s" % missing)


label balance by partition
  train  adversarial 45.2%
  val    adversarial 44.4%
  test   adversarial 44.7%
  max drift from overall 45.0% : 0.58 pp -> PASS

vector coverage in test
  misinformation                 179
  override                       182

category coverage in test
  allergy_suppression             86
  contraindication_override       89
  dosage_manipulation             96
  recommendation_alteration       90

encoding coverage in test
  base64                          23
  hex                             19   <-- thin
  leet                            16   <-- thin
  plaintext                      268
  unicode                         15   <-- thin
  url                             20


In [6]:
# ── Cell 6 ─────────────────────────


from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.dummy import DummyClassifier

def length_only_probe(train_part, test_part, label=""):
    Xtr = np.array([[d["word_count"]] for d in train_part], dtype=float)
    ytr = np.array([d["label"] for d in train_part])
    Xte = np.array([[d["word_count"]] for d in test_part], dtype=float)
    yte = np.array([d["label"] for d in test_part])

    clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
    prob = clf.predict_proba(Xte)[:, 1]
    auc  = roc_auc_score(yte, prob)
    acc  = accuracy_score(yte, clf.predict(Xte))

    base = DummyClassifier(strategy="most_frequent").fit(Xtr, ytr)
    base_acc = accuracy_score(yte, base.predict(Xte))
    return {"auc": float(auc), "accuracy": float(acc),
            "majority_baseline": float(base_acc), "n_test": int(len(yte))}

probe_v2 = length_only_probe(parts["train"], parts["test"])
print("LENGTH-ONLY PROBE — v2 corpus")
print("  A model given nothing but word count, nothing else.")
print("  AUC                : %.4f   (0.50 = no signal)" % probe_v2["auc"])
print("  accuracy           : %.4f" % probe_v2["accuracy"])
print("  majority baseline  : %.4f" % probe_v2["majority_baseline"])
verdict_v2 = "PASS" if probe_v2["auc"] < 0.60 else "FAIL - confound still present"
print("  -> %s" % verdict_v2)


rng_cf = np.random.default_rng(SEED)
n_cf = 2000
benign_cf = rng_cf.lognormal(np.log(2648), 0.6, n_cf)
attack_cf = rng_cf.lognormal(np.log(20),   0.6, n_cf)
cf_train = ([{"word_count": w, "label": 0} for w in benign_cf[:1500]] +
            [{"word_count": w, "label": 1} for w in attack_cf[:1500]])
cf_test  = ([{"word_count": w, "label": 0} for w in benign_cf[1500:]] +
            [{"word_count": w, "label": 1} for w in attack_cf[1500:]])
probe_v1 = length_only_probe(cf_train, cf_test)

print("")
print("LENGTH-ONLY PROBE — v1 design, reconstructed (2,648 vs 20 word medians)")
print("  AUC                : %.4f" % probe_v1["auc"])
print("  accuracy           : %.4f" % probe_v1["accuracy"])
print("")
print("  Interpretation: under v1's corpus design, word count alone separates")
print("  the classes almost perfectly. Under v2, it carries no signal.")
print("  v1 AUC %.4f  ->  v2 AUC %.4f" % (probe_v1["auc"], probe_v2["auc"]))


src_fields = {d.get("section") for d in dataset}
print("")
print("SOURCE PROBE")
print("  benign and adversarial chunks share one corpus (openFDA) : True")
print("  label sections represented                               : %d" % len(src_fields))
print("  -> source/label confounding removed by construction")


LENGTH-ONLY PROBE — v2 corpus
  A model given nothing but word count, nothing else.
  AUC                : 0.4862   (0.50 = no signal)
  accuracy           : 0.5527
  majority baseline  : 0.5527
  -> PASS

LENGTH-ONLY PROBE — v1 design, reconstructed (2,648 vs 20 word medians)
  AUC                : 1.0000
  accuracy           : 1.0000

  Interpretation: under v1's corpus design, word count alone separates
  the classes almost perfectly. Under v2, it carries no signal.
  v1 AUC 1.0000  ->  v2 AUC 0.4862

SOURCE PROBE
  benign and adversarial chunks share one corpus (openFDA) : True
  label sections represented                               : 8
  -> source/label confounding removed by construction


In [7]:
# ── Cell 7 ──────────────────────────────────────────────────────
def write_jsonl(p, rows):
    with open(p, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    return p

ds_path = write_jsonl(DIRS["dataset"] / "attack_dataset_split.jsonl", dataset)


group_map = {}
for d in dataset:
    group_map[d["group_id"]] = d["split"]
with open(DIRS["splits"] / "group_assignments.json", "w") as f:
    json.dump(group_map, f, indent=2, sort_keys=True)

split_stats = {
    "notebook": "03_leakage_safe_split",
    "seed": SEED,
    "method": "StratifiedGroupKFold on group_id (one group = one source document)",
    "ratios": CONFIG["SPLIT"],
    "partitions": {
        s: {
            "chunks": len(parts[s]),
            "groups": len({d["group_id"] for d in parts[s]}),
            "adversarial": int(sum(d["label"] for d in parts[s])),
            "benign": int(len(parts[s]) - sum(d["label"] for d in parts[s])),
            "adversarial_pct": float(100 * sum(d["label"] for d in parts[s]) / len(parts[s])),
        } for s in ["train", "val", "test"]
    },
    "leakage_checks": {
        "groups_spanning_partitions": len(spanning),
        "documents_spanning_partitions": len(doc_span),
        "all_chunks_assigned": len(split_of) == len(dataset),
    },
    "confound_probes": {
        "length_only_v2": probe_v2,
        "length_only_v1_counterfactual": probe_v1,
        "verdict": verdict_v2,
    },
    "test_coverage": {
        f: dict(Counter(str(d[f]) for d in parts["test"] if d["label"] == 1))
        for f in ["vector", "category", "encoding"]
    },
    "generated_at": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
}
with open(DIRS["stats"] / "split_stats.json", "w") as f:
    json.dump(split_stats, f, indent=2)

if IN_COLAB and DRIVE_ROOT:
    for sub in ["dataset", "stats", "splits"]:
        dst = DRIVE_ROOT / "data" / sub
        dst.mkdir(parents=True, exist_ok=True)
        for f_ in (ROOT / "data" / sub).glob("*"):
            shutil.copy2(f_, dst / f_.name)
    print("mirrored to Drive")

if IN_COLAB and CONFIG["PUSH_TO_GITHUB"] and REPO_DIR and REPO_DIR.exists():
    keep = ["data/stats/split_stats.json", "data/splits/group_assignments.json"]
    subprocess.run(["git", "-C", str(ROOT), "add", "-f"] + keep, check=False)
    st = subprocess.run(["git", "-C", str(ROOT), "status", "--porcelain"],
                        capture_output=True, text=True)
    if st.stdout.strip():
        msg = ("NB03: leakage-safe split, length-only AUC %.4f" % probe_v2["auc"])
        subprocess.run(["git", "-C", str(ROOT), "commit", "-q", "-m", msg], check=False)
        pr = subprocess.run(["git", "-C", str(ROOT), "push", "-q", "origin",
                             CONFIG["GITHUB_BRANCH"]], capture_output=True, text=True)
        print("push:", "ok" if pr.returncode == 0 else pr.stderr[:200])

print("written: %s (%.1f MB)" % (ds_path.name, ds_path.stat().st_size/1024/1024))
print("written: group_assignments.json (%d groups)" % len(group_map))


mirrored to Drive
push: ok
written: attack_dataset_split.jsonl (10.2 MB)
written: group_assignments.json (1487 groups)


In [8]:
# ── Cell 8 ─────────────────────────────────
print("=" * 62)
print("NB03 — LEAKAGE-SAFE SPLIT")
print("=" * 62)
print("method : StratifiedGroupKFold on group_id")
print("seed   : %d" % SEED)
print("")
print("partition    chunks   groups   adversarial")
for s in ["train", "val", "test"]:
    p = parts[s]
    print("  %-8s %6d   %6d   %5d (%.1f%%)"
          % (s, len(p), len({d["group_id"] for d in p}),
             sum(d["label"] for d in p),
             100*sum(d["label"] for d in p)/len(p)))
print("")
print("LEAKAGE CHECKS (R1-e)")
print("  groups spanning partitions    : %d  -> %s"
      % (len(spanning), "PASS" if not spanning else "FAIL"))
print("  documents spanning partitions : %d  -> %s"
      % (len(doc_span), "PASS" if not doc_span else "FAIL"))
print("  all chunks assigned           : %s" % (len(split_of) == len(dataset)))
print("")
print("CONFOUND DIAGNOSTIC (R4-6)")
print("  length-only AUC, v2 corpus        : %.4f" % probe_v2["auc"])
print("  length-only accuracy, v2          : %.4f" % probe_v2["accuracy"])
print("  majority-class baseline           : %.4f" % probe_v2["majority_baseline"])
print("  length-only AUC, v1 counterfactual: %.4f" % probe_v1["auc"])
print("  verdict                           : %s" % verdict_v2)
print("")
print("TEST COVERAGE (adversarial only)")
for f in ["vector", "category", "encoding"]:
    cnt = Counter(str(d[f]) for d in parts["test"] if d["label"] == 1)
    print("  %s:" % f)
    for k, v in sorted(cnt.items(), key=lambda x: -x[1]):
        print("    %-28s %5d" % (k, v))
print("")
print("PROBE SET BUDGET (NB04, Option C)")
print("  test chunks available as hosts : %d" % len(parts["test"]))
print("=" * 62)
print("NB03 COMPLETE — ready for NB04 (Module 1 + encoding probe set)")
print("=" * 62)


NB03 — LEAKAGE-SAFE SPLIT
method : StratifiedGroupKFold on group_id
seed   : 42

partition    chunks   groups   adversarial
  train      3787     1024    1710 (45.2%)
  val         759      228     337 (44.4%)
  test        807      235     361 (44.7%)

LEAKAGE CHECKS (R1-e)
  groups spanning partitions    : 0  -> PASS
  documents spanning partitions : 0  -> PASS
  all chunks assigned           : True

CONFOUND DIAGNOSTIC (R4-6)
  length-only AUC, v2 corpus        : 0.4862
  length-only accuracy, v2          : 0.5527
  majority-class baseline           : 0.5527
  length-only AUC, v1 counterfactual: 1.0000
  verdict                           : PASS

TEST COVERAGE (adversarial only)
  vector:
    override                       182
    misinformation                 179
  category:
    dosage_manipulation             96
    recommendation_alteration       90
    contraindication_override       89
    allergy_suppression             86
  encoding:
    plaintext                      268
   